[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_20_causal_attention_padded.ipynb)

# 🔴 Hard: Causal Attention with Padding

*Attention & Transformers*
Causal attention, but the keys are a padded batch — so some of them are not
real tokens, and some query rows end up with **nothing to attend to at all**.

### Signature
```python
def causal_attention_padded(Q, K, V, key_padding_mask=None):
    ...
```

| | shape | |
|---|---|---|
| `Q` | `(B, H, seq_q, d_k)` | |
| `K` | `(B, H, seq_k, d_k)` | `seq_k >= seq_q` |
| `V` | `(B, H, seq_k, d_v)` | `d_v` need not equal `d_k` |
| `key_padding_mask` | `(B, seq_k)` bool or `None` | `True` = real token |
| returns | `(B, H, seq_q, d_v)` | |

### 1. The causal mask is not `tril` of a square
The queries are the **last** `seq_q` positions of the sequence — that is the
decode-time convention. With `past = seq_k - seq_q`, query `i` sits at absolute
position `past + i`, so it may attend to keys `j <= past + i`:

$$j - i \le \text{past}
\quad\Longrightarrow\quad
\texttt{jnp.tril(jnp.ones((seq\_q, seq\_k)), k=past)}$$

The `k=` argument of `tril`/`triu` is exactly this threshold — `tril(m, k)`
keeps `(i, j)` iff `j - i <= k`. Drop it and you get the top-left triangle,
which hides every past key and does so **silently**:

```
seq_q=2  seq_k=5  past=3

k=past (right)        k omitted (wrong)
  q0  1 1 1 1 0         q0  1 0 0 0 0
  q1  1 1 1 1 1         q1  1 1 0 0 0
```

### 2. The padding mask lives on a different axis
Causal is `(seq_q, seq_k)` — a property of *positions*. Padding is
`(B, seq_k)` — a property of *batch items*. They meet on a `(B, H, seq_q,
seq_k)` score array, so the padding mask needs its head and query axes
inserted before the two can be combined with `&`.

### 3. Some rows have nothing left, and that is the real problem
A query whose whole causal window is padding has **no visible key**. Softmax
over an all-blocked row cannot produce a distribution, and neither fill value
saves you:

```
fill = -inf   ->  [nan nan nan nan nan]                     poisons everything
fill = -1e9   ->  [0.237, 0.299, -0.355, 0.149, -0.047]     finite, plausible, garbage
```

The `-1e9` case is the dangerous one: softmax of equal logits is **uniform**,
so the row returns the average of the padding vectors. No error, no NaN, and a
number that looks perfectly reasonable.

Return **exactly zero** for such rows. That means finding them —
`jnp.any(allowed, axis=-1)` — and zeroing after the softmax; a fill value alone
cannot express "no answer".

### Everything else
Scale by $\sqrt{d_k}$ (not $d_v$). With `key_padding_mask=None` and
`seq_q == seq_k` this must reduce exactly to ordinary causal self-attention.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def causal_attention_padded(Q, K, V, key_padding_mask=None):
    """Causal attention over a padded batch of keys.

    Args:
        Q: (B, H, seq_q, d_k)
        K: (B, H, seq_k, d_k)   seq_k >= seq_q
        V: (B, H, seq_k, d_v)
        key_padding_mask: (B, seq_k) bool, True = real token, or None

    Returns:
        (B, H, seq_q, d_v). Rows with no visible key are exactly zero.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

B, H, seq_q, seq_k, d_k, d_v = 1, 1, 2, 5, 4, 3
k = jax.random.split(jax.random.key(0), 3)
Q = jax.random.normal(k[0], (B, H, seq_q, d_k))
K = jax.random.normal(k[1], (B, H, seq_k, d_k))
V = jax.random.normal(k[2], (B, H, seq_k, d_v))

past = seq_k - seq_q
print(f"past = seq_k - seq_q = {past}, so the causal mask is tril(k={past}):")
print(jnp.tril(jnp.ones((seq_q, seq_k), dtype=int), k=past))

print("\nno padding:")
print(causal_attention_padded(Q, K, V))

# Pad away everything query 0 could have seen (keys 0..past).
mask = jnp.ones((B, seq_k), dtype=bool).at[0, : past + 1].set(False)
out = causal_attention_padded(Q, K, V, mask)
print(f"\nkey_padding_mask = {mask[0]}")
print("query 0 now has no visible key, so its row must be exactly zero:")
print(out)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("causal_attention_padded")

# hint("causal_attention_padded")      # stuck? nudge without the answer
# solution("causal_attention_padded")  # spoiler: the reference implementation
# status()                             # your dashboard across all problems